In [6]:
import torch
import pandas as pd
import torch.nn as nn
from torch.nn import functional as F
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

In [7]:
torch.manual_seed(37)
torch.cuda.manual_seed_all(37)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

In [8]:
class NGramDataset(Dataset):
    def __init__(self, data, vocab, context_size):
        self.vocab = vocab
        self.context_size = context_size
        self.stoi = {s:i for i, s in enumerate(self.vocab)}
        self.itos = {i:s for s, i in self.stoi.items()}
        self.X, self.Y = self.construct_dataset(data, self.context_size)
    def construct_dataset(self, data, context_size):
        X, Y = [], []
        for w in data:
            context = [self.stoi["^"]] * context_size
            for ch in w + "$":
                ix = self.stoi[ch]
                X.append(context)
                Y.append(ix)
                context = context[1:] + [ix]
        X = torch.tensor(X)
        Y = torch.tensor(Y)
        return X, Y
    def __len__(self):
        return self.X.shape[0]
    def get_vocab_size(self):
        return len(self.vocab)
    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

In [9]:
def create_dataset(input_file, weird_symbol='"', split_factor = 0.1, context_size = 1):
    data = pd.read_csv(input_file, sep=";")["NAZWA"].drop_duplicates().str.lower()
    vocab = sorted(list(set("".join(data))))
    SOS = "^"
    EOS = "$"
    vocab = [SOS] + vocab + [EOS]
    max_length = data.str.len().max()
    max_length_word = data[data.str.len() == data.str.len().max()]
    print(f"Number of examples in the dataset: {data.shape}\n")
    print(f"\nMax word length: {max_length}")
    print(f"Max length word: {max_length_word.values} \n")
    print(f"\nWeird symbol lookup: {data[data.str.contains(weird_symbol, regex=False)].values}\n")
    print(f"\nNumber of unique characters in the vocabulary: {len(vocab)}")
    print("Vocabulary:")
    print(''.join(vocab))

    test_set_size = int(max(1000, data.shape[0] * split_factor))
    rp = torch.randperm(data.shape[0]).tolist()
    test_set = [data.iloc[i] for i in rp[:test_set_size]]
    cv_set = [data.iloc[i] for i in rp[test_set_size:2*test_set_size]]
    train_set = [data.iloc[i] for i in rp[2*test_set_size:]]
    print(f"\nSplit up the dataset into {len(train_set)} training examples, {len(cv_set)} cross validation examples and {len(test_set)} cross validation examples")

    train = NGramDataset(train_set, vocab, context_size)
    val = NGramDataset(cv_set, vocab, context_size)
    test = NGramDataset(test_set, vocab, context_size)
    return train, val, test

In [10]:
train_dataset, val_dataset, test_dataset = create_dataset("SIMC_Urzedowy_2026-01-18.csv", context_size=12)

Number of examples in the dataset: (57692,)


Max word length: 40
Max length word: ['wólka sokołowska k. wólki niedźwiedzkiej'] 


Weird symbol lookup: ['mazewo dworskie"b"' 'mazewo dworskie"a"']


Number of unique characters in the vocabulary: 39
Vocabulary:
^ "-.abcdefghijklmnoprstuvwyzóąćęłńśźż$

Split up the dataset into 46154 training examples, 5769 cross validation examples and 5769 cross validation examples


In [11]:
train_dataloader = DataLoader(train_dataset, batch_size=128, shuffle=True)
val_dataloader = DataLoader(val_dataset, batch_size=128, shuffle=False)
test_dataloader = DataLoader(test_dataset, batch_size=128, shuffle=False)

In [12]:
class NGramRNN(nn.Module):
    def __init__(self, vocab_size, embed_size, hidden_size, output_size, num_layers=2):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_size)
        self.gru = nn.GRU(embed_size, hidden_size, num_layers=num_layers, batch_first=True, dropout=0.3 if num_layers > 1 else 0)
        self.fc1 = nn.Linear(hidden_size, output_size)
    def forward(self, x):
        x = self.embedding(x)
        _, h_n = self.gru(x)
        h_n = h_n[-1]
        out = self.fc1(h_n)
        return out

In [13]:
vocab_size = train_dataset.get_vocab_size()
embed_size = 128
hidden_size = 256
output_size = train_dataset.get_vocab_size()
model = NGramRNN(vocab_size, embed_size, hidden_size, output_size)

In [14]:
model.to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-2)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, 'min', patience=3, factor=0.5)

In [15]:
def evaluate(model, dataloader, device, top_k=3):
    model.eval()
    total_loss = 0.0
    total_correct = 0
    total_topk_correct = 0
    total_samples = 0

    with torch.no_grad():
        for x, y in dataloader:
            x = x.to(device)
            y = y.to(device)

            logits = model(x)
            loss = F.cross_entropy(logits, y)

            total_loss += loss.item() * x.size(0)

            preds = logits.argmax(dim=1)
            total_correct += (preds == y).sum().item()

            topk_preds = logits.topk(top_k, dim=1).indices
            total_topk_correct += (topk_preds == y.unsqueeze(1)).any(dim=1).sum().item()

            total_samples += x.size(0)

    avg_loss = total_loss / total_samples
    top1_acc = total_correct / total_samples
    topk_acc = total_topk_correct / total_samples

    return avg_loss, top1_acc, topk_acc

In [16]:
print(evaluate(model, val_dataloader, device, top_k=3))
val_dataset.get_vocab_size()
-torch.log(torch.tensor([1/39]))

(3.6588853115106224, 0.04080438531658044, 0.10376632175412663)


tensor([3.6636])

In [17]:
EPOCHS = 25

for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    total_correct = 0
    total_topk_correct = 0
    total_samples = 0

    for x, y in tqdm(train_dataloader, desc=f"Epoch {epoch + 1}"):
        x = x.to(device)
        y = y.to(device)

        optimizer.zero_grad()
        logits = model(x)
        loss = F.cross_entropy(logits, y)
        loss.backward()
        optimizer.step()

        total_loss += loss.item() * x.size(0)

        preds = logits.argmax(dim=1)
        total_correct += (preds == y).sum().item()

        topk_preds = logits.topk(3, dim=1).indices
        total_topk_correct += (topk_preds == y.unsqueeze(1)).any(dim=1).sum().item()

        total_samples += x.size(0)

    avg_train_loss = total_loss / total_samples
    train_top1 = total_correct / total_samples
    train_top3 = total_topk_correct / total_samples

    val_loss, val_top1, val_top3 = evaluate(model, val_dataloader, device, top_k=3)
    scheduler.step(val_loss)

    print(f"Epoch {epoch + 1}: "
          f"train loss = {avg_train_loss:.4f}, train top-1 = {train_top1:.4f}, train top-3 = {train_top3:.4f}, "
          f"val loss = {val_loss:.4f}, val top-1 = {val_top1:.4f}, val top-3 = {val_top3:.4f}")

Epoch 1: 100%|██████████| 4062/4062 [00:18<00:00, 221.27it/s]


Epoch 1: train loss = 1.8568, train top-1 = 0.4209, train top-3 = 0.6778, val loss = 1.7037, val top-1 = 0.4666, val top-3 = 0.7154


Epoch 2: 100%|██████████| 4062/4062 [00:17<00:00, 227.67it/s]


Epoch 2: train loss = 1.6882, train top-1 = 0.4676, train top-3 = 0.7164, val loss = 1.6605, val top-1 = 0.4784, val top-3 = 0.7169


Epoch 3: 100%|██████████| 4062/4062 [00:18<00:00, 214.02it/s]


...

Epoch 23: train loss = 1.5414, train top-1 = 0.5061, train top-3 = 0.7493, val loss = 1.5850, val top-1 = 0.4994, val top-3 = 0.7437


Epoch 24: 100%|██████████| 4062/4062 [00:18<00:00, 224.06it/s]


Epoch 24: train loss = 1.5409, train top-1 = 0.5066, train top-3 = 0.7499, val loss = 1.5831, val top-1 = 0.4963, val top-3 = 0.7432


Epoch 25: 100%|██████████| 4062/4062 [00:17<00:00, 228.11it/s]


Epoch 25: train loss = 1.5394, train top-1 = 0.5060, train top-3 = 0.7499, val loss = 1.5798, val top-1 = 0.5008, val top-3 = 0.7383


In [18]:
test_loss, test_top1, test_top3 = evaluate(model, test_dataloader, device, top_k=3)
print(f"test loss = {test_loss:.4f}, train top-1 = {test_top1:.4f}, train top-3 = {test_top3:.4f}")

test loss = 1.5849, train top-1 = 0.4965, train top-3 = 0.7366


In [19]:
def generate(model, stoi, itos, context_size=12, max_length=20, device='cpu', temperature = 1):
    model.eval()
    context = [stoi["^"]] * context_size
    result = []

    with torch.no_grad():
        for _ in range(max_length):
            x = torch.tensor([context], dtype=torch.long).to(device)
            logits = model(x)
            probs = F.softmax(logits, dim=-1)
            ix = torch.multinomial(probs, num_samples=1).item()
            if itos[ix] == "$":
                break
            result.append(itos[ix])
            context = context[1:] + [ix]

    return "".join(result)

In [20]:
stoi = train_dataset.stoi
itos = train_dataset.itos
for _ in range(15):
    city = generate(model, stoi, itos, context_size=train_dataset.context_size, max_length=15, device=device, temperature = 1)
    print(city)

morek
kolonia lasowsk
piętaśnika
łabędzin
stanygóra
lubińsk
brównik
wielka strona
gojęzewo
kwaśniak
łupaszyn
święte górki
ciechanka nowa
przełęczyn
wólka


In [21]:
# RNN
# BASELINE: context_size=1; embed_size = 32; hidden_size = 32; num_layers = 1; batch_size=128; lr=1e-4
# Epoch 20: train loss = 2.4161, train top-1 = 0.2508, train top-3 = 0.5297, val loss = 2.4208, val top-1 = 0.2492, val top-3 = 0.5294

# OVERFIT: context_size=8; embed_size = 256; hidden_size = 256; num_layers = 4; batch_size=128; lr=1e-4
# Epoch 20: train loss = 1.4388, train top-1 = 0.5356, train top-3 = 0.7754, val loss = 1.5901, val top-1 = 0.4945, val top-3 = 0.7440

# OVERFIT: context_size=8; embed_size = 256; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3
# Epoch 20: train loss = 1.5634, train top-1 = 0.4967, train top-3 = 0.7479, val loss = 1.6426, val top-1 = 0.4792, val top-3 = 0.7252

# LEKKI OVERFIT: context_size=10; embed_size = 256; hidden_size = 256; num_layers = 1; batch_size=128; lr=1e-4
# Epoch 20: train loss = 1.5940, train top-1 = 0.4934, train top-3 = 0.7414, val loss = 1.6522, val top-1 = 0.4801, val top-3 = 0.7277
#
#
#


In [22]:
# GRU
# BASELINE: context_size=1; embed_size = 32; hidden_size = 32; num_layers = 1; batch_size=128; lr=1e-4
# Epoch 20: train loss = 2.4154, train top-1 = 0.2510, train top-3 = 0.5295, val loss = 2.4200, val top-1 = 0.2498, val top-3 = 0.5296

# LEKKI OVERFIT: context_size=8; embed_size = 128; hidden_size = 256; num_layers = 1; batch_size=128; lr=1e-4
# Epoch 20: train loss = 1.5245, train top-1 = 0.5133, train top-3 = 0.7571, val loss = 1.6070, val top-1 = 0.4911, val top-3 = 0.7376

# context_size=8; embed_size = 128; hidden_size = 128; num_layers = 1; batch_size=128; lr=1e-4
# Epoch 20: train loss = 1.6515, train top-1 = 0.4788, train top-3 = 0.7291, val loss = 1.6880, val top-1 = 0.4690, val top-3 = 0.7195

# context_size=8; embed_size = 128; hidden_size = 256; num_layers = 1; batch_size=128; lr=1e-3
# Epoch 20: train loss = 1.4338, train top-1 = 0.5350, train top-3 = 0.7753, val loss = 1.6181, val top-1 = 0.4913, val top-3 = 0.7391

# context_size=8; embed_size = 128; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.2
# Epoch 20: train loss = 1.5271, train top-1 = 0.5079, train top-3 = 0.7546, val loss = 1.6009, val top-1 = 0.4917, val top-3 = 0.7397

# context_size=8; embed_size = 128; hidden_size = 512; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.2
# Epoch 20: train loss = 1.5946, train top-1 = 0.4896, train top-3 = 0.7405, val loss = 1.6454, val top-1 = 0.4786, val top-3 = 0.7263

# context_size=8; embed_size = 128; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.2 + wd + scheduler
# Epoch 19: train loss = 1.5276, train top-1 = 0.5074, train top-3 = 0.7551, val loss = 1.5977, val top-1 = 0.4949, val top-3 = 0.7437
# Epoch 20: train loss = 1.4791, train top-1 = 0.5214, train top-3 = 0.7658, val loss = 1.5697, val top-1 = 0.5016, val top-3 = 0.7454

# context_size=12; embed_size = 128; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.2 + wd + scheduler
# Epoch 20: train loss = 1.4243, train top-1 = 0.5384, train top-3 = 0.7748, val loss = 1.5418, val top-1 = 0.5104, val top-3 = 0.7496
# Epoch 50: train loss = 1.2655, train top-1 = 0.5865, train top-3 = 0.8043, val loss = 1.5432, val top-1 = 0.5185, val top-3 = 0.7571

# context_size=12; embed_size = 128; hidden_size = 384; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.3 + wd + scheduler
# Epoch 50: train loss = 1.2411, train top-1 = 0.5928, train top-3 = 0.8081, val loss = 1.5488, val top-1 = 0.5197, val top-3 = 0.7577

# context_size=12; embed_size = 128; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.3 + wd + scheduler + second linear
# Epoch 50: train loss = 1.3157, train top-1 = 0.5711, train top-3 = 0.7953, val loss = 1.5391, val top-1 = 0.5166, val top-3 = 0.7569


# context_size=12; embed_size = 128; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.3 + wd + scheduler
# Epoch 25: train loss = 1.5394, train top-1 = 0.5060, train top-3 = 0.7499, val loss = 1.5798, val top-1 = 0.5008, val top-3 = 0.7383

# context_size=12; embed_size = 128; hidden_size = 256; num_layers = 2; batch_size=128; lr=1e-3 + DROPOUT 0.3 + wd + scheduler + label smoothing
# Epoch 25: train loss = 1.6950, train top-1 = 0.5351, train top-3 = 0.7716, val loss = 1.5549, val top-1 = 0.5128, val top-3 = 0.7523
